# SQL Agent Workflow

Chinook 数据库上的完整链路：

数据库解析 → table 选择 → SQL 生成 → SQL 校验 → SQL 改写 → SQL 执行 → 结果汇总

In [ ]:
import os, re, sqlite3
from openai import OpenAI

DB = r"E:\BaiduNetdiskDownload\ai学习\第11周-ChatBI数据智能问答\Week11\04_SQL-Code-Agent-Demo\chinook.db"
MODEL = "qwen-max"

client = OpenAI(
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

In [ ]:
class DBSchema:
    def __init__(self, path):
        self.conn = sqlite3.connect(path)
        self.cur = self.conn.cursor()
        self.tables = [r[0] for r in self.cur.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")]
        self._detail = {t: self._describe(t) for t in self.tables}
        self.detail = "\n\n".join(self._detail.values())
        self.brief = "\n".join(
            f"{t}: {', '.join(c[1] for c in self.cur.execute(f'PRAGMA table_info("{t}")'))}"
            for t in self.tables)

    def _describe(self, table):
        lines = [f"表 {table}"]
        cols = self.cur.execute(f'PRAGMA table_info("{table}")').fetchall()
        for _, name, ctype, *_ in cols:
            vals = [r[0] for r in self.cur.execute(
                f'SELECT DISTINCT "{name}" FROM "{table}" WHERE "{name}" IS NOT NULL LIMIT 3')]
            lines.append(f"  {name} {ctype} 示例 {vals}")
        for fk in self.cur.execute(f'PRAGMA foreign_key_list("{table}")').fetchall():
            lines.append(f"  外键 {fk[3]} -> {fk[2]}.{fk[4]}")
        return "\n".join(lines)

    def schema_of(self, tables):
        names = [t for t in re.split(r"[,，\s]+", tables) if t in self._detail]
        return "\n\n".join(self._detail[t] for t in names) or self.detail

    def check(self, sql):
        try:
            self.cur.execute("EXPLAIN " + sql)
            return True, ""
        except Exception as e:
            return False, str(e)

    def run(self, sql):
        cur = self.conn.execute(sql)
        return [d[0] for d in cur.description], cur.fetchall()

In [ ]:
db = DBSchema(DB)
print(db.tables)
print(db.detail[:600])

In [ ]:
def ask(prompt):
    r = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    return r.choices[0].message.content.strip()


def clean_sql(text):
    text = text.strip().strip("`")
    return re.sub(r"^sql\s*\n", "", text, flags=re.I).strip()

In [ ]:
def select_tables(question):
    prompt = f"""从下列表中选出回答该问题所需的表，只输出表名，多个用逗号分隔，不要解释。

{db.brief}

问题：{question}"""
    return ask(prompt)


def generate_sql(question, tables):
    prompt = f"""你是SQLite专家。根据表结构生成一条SQL，只输出SQL。

{db.schema_of(tables)}

问题：{question}"""
    return clean_sql(ask(prompt))

In [ ]:
def rewrite_sql(question, sql, err):
    prompt = f"""下面的SQL在SQLite执行报错，请修正，只输出修正后的SQL。

{db.detail}

SQL：{sql}
报错：{err}
问题：{question}"""
    return clean_sql(ask(prompt))


def summarize(question, cols, rows):
    prompt = f"""根据查询结果回答问题，用简洁的自然语言，不要提SQL。

问题：{question}
结果列：{cols}
结果行：{rows[:30]}"""
    return ask(prompt)

In [ ]:
def run_sql_agent(question, max_retry=3):
    # trace 是中间过程的记录器：它保存每一步的输入输出，用于事后排查和评估。
    trace = {}
    tables = select_tables(question)
    trace["tables"] = tables

    sql = generate_sql(question, tables)
    trace["sql_initial"] = sql

    for i in range(max_retry):
        ok, err = db.check(sql)
        if ok:
            break
        trace[f"error_{i + 1}"] = err
        sql = rewrite_sql(question, sql, err)
        trace[f"sql_rewrite_{i + 1}"] = sql

    cols, rows = db.run(sql)
    answer = summarize(question, cols, rows)
    return {"answer": answer, "sql": sql, "rows": rows, "trace": trace}

In [ ]:
for q in ["一共有多少位艺术家？", "哪个国家的客户最多？", "销售额最高的前5位艺术家是谁？"]:
    r = run_sql_agent(q)
    print("Q:", q)
    print("表:", r["trace"]["tables"])
    print("SQL:", r["sql"])
    print("A:", r["answer"])
    print("-" * 70)